In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/online_retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Snapshot date = one day after the last transaction in the data
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

Snapshot date: 2011-12-10 12:50:00


In [2]:
recency_df = df.groupby('Customer ID')['InvoiceDate'].max().reset_index()
recency_df.columns = ['Customer ID', 'LastPurchaseDate']

recency_df['Recency'] = (snapshot_date - recency_df['LastPurchaseDate']).dt.days

recency_df.head(10)

,Customer ID,LastPurchaseDate,Recency
0,12346.0,2011-01-18 10:01:00,326
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,19
4,12350.0,2011-02-02 16:01:00,310
5,12351.0,2010-11-29 15:23:00,375
6,12352.0,2011-11-03 14:37:00,36
7,12353.0,2011-05-19 17:47:00,204
8,12354.0,2011-04-21 13:11:00,232
9,12355.0,2011-05-09 13:49:00,214


In [3]:
frequency_df = df.groupby('Customer ID')['Invoice'].nunique().reset_index()
frequency_df.columns = ['Customer ID', 'Frequency']

frequency_df.head(10)

,Customer ID,Frequency
0,12346.0,12
1,12347.0,8
2,12348.0,5
3,12349.0,4
4,12350.0,1
5,12351.0,1
6,12352.0,10
7,12353.0,2
8,12354.0,1
9,12355.0,2


In [4]:
df['TotalPrice'] = df['Quantity'] * df['Price']

monetary_df = df.groupby('Customer ID')['TotalPrice'].sum().reset_index()
monetary_df.columns = ['Customer ID', 'Monetary']
monetary_df['Monetary'] = monetary_df['Monetary'].round(2)

monetary_df.head(10)


,Customer ID,Monetary
0,12346.0,77556.46
1,12347.0,5633.32
2,12348.0,2019.40
3,12349.0,4428.69
4,12350.0,334.40
5,12351.0,300.93
6,12352.0,2849.84
7,12353.0,406.76
8,12354.0,1079.40
9,12355.0,947.61


In [5]:
rfm = recency_df[['Customer ID', 'Recency']].merge(
    frequency_df, on='Customer ID'
).merge(
    monetary_df, on='Customer ID'
)

print(rfm.shape)
rfm.head(10)

(5878, 4)


,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40
5,12351.0,375,1,300.93
6,12352.0,36,10,2849.84
7,12353.0,204,2,406.76
8,12354.0,232,1,1079.40
9,12355.0,214,2,947.61


In [6]:
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,12346.0,326,12,77556.46,2,5,5
1,12347.0,2,8,5633.32,5,4,5
2,12348.0,75,5,2019.40,3,4,4
3,12349.0,19,4,4428.69,5,3,5
4,12350.0,310,1,334.40,2,1,2
5,12351.0,375,1,300.93,2,1,2
6,12352.0,36,10,2849.84,4,5,4
7,12353.0,204,2,406.76,2,2,2
8,12354.0,232,1,1079.40,2,1,3
9,12355.0,214,2,947.61,2,2,3


In [7]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,12346.0,326,12,77556.46,2,5,5,12
1,12347.0,2,8,5633.32,5,4,5,14
2,12348.0,75,5,2019.40,3,4,4,11
3,12349.0,19,4,4428.69,5,3,5,13
4,12350.0,310,1,334.40,2,1,2,5
5,12351.0,375,1,300.93,2,1,2,5
6,12352.0,36,10,2849.84,4,5,4,13
7,12353.0,204,2,406.76,2,2,2,6
8,12354.0,232,1,1079.40,2,1,3,6
9,12355.0,214,2,947.61,2,2,3,7


In [8]:
def segment_customer(score):
    if score >= 13:
        return 'Champions'
    elif score >= 10:
        return 'Loyal Customers'
    elif score >= 7:
        return 'At Risk'
    elif score >= 4:
        return 'Hibernating'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(segment_customer)

rfm['Segment'].value_counts()

Segment
Hibernating        1456
At Risk            1450
Loyal Customers    1355
Champions          1297
Lost                320
Name: count, dtype: int64

In [9]:
segment_summary = rfm.groupby('Segment').agg(
    num_customers=('Customer ID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    total_revenue=('Monetary', 'sum')
).round(1).sort_values('total_revenue', ascending=False)

segment_summary

,num_customers,avg_recency,avg_frequency,avg_monetary,total_revenue
Segment,,,,,
Champions,1297,25.9,17.9,9878.0,12811734.5
Loyal Customers,1355,96.0,5.6,2310.0,3129997.0
At Risk,1450,202.1,2.7,868.9,1259965.5
Hibernating,1456,373.1,1.3,339.9,494874.5
Lost,320,573.6,1.0,146.4,46857.6


In [10]:
# Save RFM table to CSV
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)

# Also load it into the SQLite database so it's queryable alongside transactions
import sqlite3
conn = sqlite3.connect('../data/processed/retail.db')
rfm.to_sql('rfm_segments', conn, if_exists='replace', index=False)

print("Saved RFM table to CSV and SQLite.")

Saved RFM table to CSV and SQLite.
